# 03 · Event windows

**Goal:** for each of the 13 events, measure what USD/INR and the other markets did over eight trading-day windows,
how unusual each move was, and what else was happening at the same time.

> **Series:** 01 Ingest → 02 Align & transform → 03 Event windows → 04 Database → 05 Results report → 06 Analysis & charts. Run them in order (or run `python run_pipeline.py` from the project folder). Paths and settings live in `config.py`.

| Window | From → To | Meaning |
|---|---|---|
| PRE_30 / 14 / 7 | t(−k) → t(−1) | run-up before the event |
| EVENT | t(−1) → t(0) | event-day reaction |
| POST_7 / 14 / 30 / 90 | t(−1) → t(+k) | cumulative move including the event day |

**t(0)** is the first trading day **on or after** the anchor date, the day the news became public to Indian markets.
**t(−1)** is the last close *before* the news.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

import numpy as np
import pandas as pd
from config import (PROC, REF_EVENTS, REF_REPO, WINDOWS, PATH_RANGE, ANNUALISE, MIN_OBS_WINDOW_CORR,
                    BASELINE_LOOKBACK, LEVEL_CHANGE_SERIES, PANEL_SERIES, CORR_PAIRS, FX_SOURCES,
                    read_reference)

WINDOW_SERIES = ["USDINR"] + PANEL_SERIES
BASELINE_SERIES = ["USDINR", "BRENT", "GOLD_USD", "NIFTY50", "DXY_BROAD"]

## The event list

In [2]:
events = read_reference(REF_EVENTS)
events[["event_id", "event_name", "event_start_date", "anchor_date", "event_end_date", "india_direct", "source_org"]]

,event_id,event_name,event_start_date,anchor_date,event_end_date,india_direct,source_org
0,E01,Russia authorises force in Ukraine / Crimea annexation,2014-03-01,2014-03-01,2014-03-18,N,Radio Free Europe/Radio Liberty
1,E02,Indian Army cross-LoC 'surgical strikes' (post-Uri),2016-09-29,2016-09-29,2016-09-29,Y,Press Information Bureau (via GlobalSecurity mirror)
2,E03,Doklam India-China standoff,2017-06-16,2017-06-30,2017-08-28,Y,Ministry of External Affairs (India)
3,E04,Pulwama attack and Balakot airstrike,2019-02-14,2019-02-15,2019-03-01,Y,Ministry of External Affairs (India)
4,E05,US strike kills IRGC commander Qassem Soleimani,2020-01-03,2020-01-03,2020-01-08,N,Military.com
5,E06,Galwan Valley clash,2020-06-15,2020-06-16,2020-06-15,Y,Zee News (quotes Indian Army statement)
6,E07,Russia's full-scale invasion of Ukraine,2022-02-24,2022-02-24,NaN,N,President of Russia (Kremlin)
7,E08,Hamas attack on Israel and Israel-Gaza war,2023-10-07,2023-10-09,NaN,N,United Nations
8,E09,Iran's first direct missile and drone attack on Israel,2024-04-13,2024-04-15,2024-04-19,N,United Nations Secretary-General
9,E10,Pahalgam attack and Operation Sindoor,2025-04-22,2025-04-23,2025-05-10,Y,Ministry of External Affairs (India)


## Loading the panels and the context data (repo changes, FOMC changes, RBI intervention)

In [3]:
def load():
    ev = read_reference(REF_EVENTS)
    for c in ["event_start_date", "event_end_date", "anchor_date"]:
        ev[c] = pd.to_datetime(ev[c])
    panels = {b: pd.read_csv(PROC / f"panel_wide_{b}.csv", parse_dates=["date"]).set_index("date")
              for b in FX_SOURCES}
    long = pd.read_csv(PROC / "panel_daily_long.csv", parse_dates=["date"])
    stale = long.pivot_table(index=["basis", "date"], columns="series_id", values="staleness_days")
    repo = read_reference(REF_REPO)
    repo["effective_date"] = pd.to_datetime(repo["effective_date"])
    repo["change_bp"] = repo["change_bp"].astype(int)
    obs = pd.read_csv(PROC / "observations_long.csv", parse_dates=["obs_date"])
    ffr = obs[obs.series_id == "FFR_UPPER"].set_index("obs_date")["value"]
    ffr_changes = ffr[ffr.diff().fillna(0) != 0]
    fxi = obs[obs.series_id == "IN_RBI_FX_NET"].set_index("obs_date")["value"]
    return ev, panels, stale, repo, ffr_changes, fxi

## Measuring a window

Prices are measured in **%**, while rates and yields are measured in **percentage points**. The **baseline** compares the
event-window move with *every* same-length move in the ~750 trading days before the window starts. The result is a
percentile, which describes how unusual the move was. It is **not a significance test**, because overlapping windows
are autocorrelated.

In [4]:
def measure(series_id, v0, v1):
    """Window change: % for prices, percentage points for rates/yields."""
    if pd.isna(v0) or pd.isna(v1):
        return np.nan, np.nan
    if series_id in LEVEL_CHANGE_SERIES:
        return v1 - v0, np.nan
    return (v1 / v0 - 1) * 100, np.log(v1 / v0)


def baseline(values, i_start, length, event_log_change):
    """Distribution of all same-length log changes in the BASELINE_LOOKBACK days
    that END at or before the window start (so the event itself is excluded)."""
    lv = np.log(values)
    ch = lv - lv.shift(length)
    lo = max(length, i_start - BASELINE_LOOKBACK)
    sample = ch.iloc[lo:i_start + 1].dropna()
    if len(sample) < 100 or pd.isna(event_log_change):
        return dict(baseline_n=len(sample))
    mu, sd = sample.mean(), sample.std()
    return dict(baseline_n=len(sample),
                baseline_mean_pct=(np.exp(mu) - 1) * 100,
                baseline_sd_pct=sd * 100,
                z_score=(event_log_change - mu) / sd if sd > 0 else np.nan,
                percentile=(sample < event_log_change).mean() * 100)

## The event-window engine

For every calendar × event × window × series, it records the start and end values, the change, window volatility,
endpoint staleness and the baseline percentile. For every window it also records **overlapping events**,
**RBI repo changes**, **FOMC changes** and **RBI net FX intervention**. Status is `complete`, or `partial` if the data ends before t(+k).

In [5]:
def compute_event_windows():
    ev, panels, stale, repo, ffr_changes, fxi = load()
    anchors, rows, corr_rows, path_rows, ctx_rows = [], [], [], [], []

    for basis, P in panels.items():
        dates = P.index
        n = len(dates)
        # anchor index of every event on this calendar (for overlap detection)
        a_idx = {}
        for _, e in ev.iterrows():
            pos = dates.searchsorted(e.anchor_date)          # first date >= anchor
            a_idx[e.event_id] = pos if pos < n else None

        for _, e in ev.iterrows():
            t0 = a_idx[e.event_id]
            if t0 is None or t0 == 0:
                anchors.append(dict(basis=basis, event_id=e.event_id, covered=0,
                                    note="anchor after end of this FX calendar"))
                continue
            anchors.append(dict(basis=basis, event_id=e.event_id, covered=1,
                                anchor_date=e.anchor_date.date(),
                                t0_date=dates[t0].date(), t_minus1_date=dates[t0 - 1].date(),
                                calendar_days_anchor_to_t0=(dates[t0] - e.anchor_date).days,
                                last_available_date=dates[-1].date(),
                                trading_days_after_available=n - 1 - t0,
                                note=""))

            for wid, wlabel, s_off, e_off, wtype, _ in WINDOWS:
                i_s, i_e_target = t0 + s_off, t0 + e_off
                if i_s < 0:
                    status, i_e = "not_covered", None
                elif i_e_target <= n - 1:
                    status, i_e = "complete", i_e_target
                else:
                    status, i_e = "partial", n - 1
                if i_e is None or i_e <= i_s:
                    rows.append(dict(basis=basis, event_id=e.event_id, window_id=wid,
                                     series_id="USDINR", status="not_covered"))
                    continue
                d_s, d_e = dates[i_s], dates[i_e]
                seg = P.iloc[i_s + 1:i_e + 1]          # daily returns inside the window
                others = [k for k, v in a_idx.items()
                          if k != e.event_id and v is not None and i_s <= v <= i_e]
                repo_in = repo[(repo.effective_date > d_s) & (repo.effective_date <= d_e)]
                ffr_in = ffr_changes[(ffr_changes.index > d_s) & (ffr_changes.index <= d_e)]
                # RBI net FX purchase(+)/sale(-): monthly data, so sum every calendar month the window touches
                m0, m1 = d_s.to_period("M").to_timestamp(), d_e.to_period("M").to_timestamp()
                fx_m = fxi[(fxi.index >= m0) & (fxi.index <= m1)]
                n_months = len(pd.period_range(m0, m1, freq="M"))
                ctx_rows.append(dict(
                    basis=basis, event_id=e.event_id, window_id=wid, start_date=d_s.date(),
                    end_date=d_e.date(), status=status,
                    trading_days=i_e - i_s,
                    overlapping_events=",".join(others),
                    rbi_repo_changes=";".join(f"{r.effective_date.date()} {r.change_bp:+d}bp"
                                              for r in repo_in.itertuples()),
                    fomc_target_changes=";".join(f"{d.date()} to {v:.2f}%" for d, v in ffr_in.items()),
                    rbi_net_fx_usd_mn=fx_m.sum() if len(fx_m) == n_months else np.nan,
                    rbi_fx_months=f"{m0:%b-%Y}..{m1:%b-%Y}" + ("" if len(fx_m) == n_months
                                                              else " (incomplete: data not yet published)")))

                for sid in WINDOW_SERIES:
                    v0, v1 = P[sid].iloc[i_s], P[sid].iloc[i_e]
                    pct, logc = measure(sid, v0, v1)
                    rcol = "USDINR_ret" if sid == "USDINR" else f"{sid}_ret"
                    r = seg[rcol].dropna()
                    st0 = st1 = 0
                    if sid != "USDINR":
                        st0 = stale.at[(basis, d_s), sid] if sid in stale.columns else np.nan
                        st1 = stale.at[(basis, d_e), sid] if sid in stale.columns else np.nan
                    row = dict(basis=basis, event_id=e.event_id, window_id=wid, series_id=sid,
                               status=status if not pd.isna(pct) else "no_data",
                               start_date=d_s.date(), end_date=d_e.date(),
                               trading_days=i_e - i_s, start_value=v0, end_value=v1,
                               change=pct, log_change=logc,
                               change_unit="pp" if sid in LEVEL_CHANGE_SERIES else "%",
                               n_returns=len(r),
                               window_vol_ann_pct=(r.std() * np.sqrt(ANNUALISE) * 100
                                                   if len(r) >= 5 and sid not in LEVEL_CHANGE_SERIES
                                                   else np.nan),
                               start_staleness_days=st0, end_staleness_days=st1)
                    if sid == "USDINR" and not pd.isna(pct):
                        row["inr_pct_change"] = (v0 / v1 - 1) * 100
                        dxy0, dxy1 = P["DXY_BROAD"].iloc[i_s], P["DXY_BROAD"].iloc[i_e]
                        if not (pd.isna(dxy0) or pd.isna(dxy1)):
                            row["usdinr_minus_dxy_pp"] = pct - (dxy1 / dxy0 - 1) * 100
                    if sid in BASELINE_SERIES and status == "complete":
                        row.update(baseline(P[sid], i_s, i_e - i_s, logc))
                    rows.append(row)

                if len(seg) >= MIN_OBS_WINDOW_CORR:
                    for sid in CORR_PAIRS:
                        pair = seg[["USDINR_ret", f"{sid}_ret"]].dropna()
                        if len(pair) >= MIN_OBS_WINDOW_CORR:
                            corr_rows.append(dict(basis=basis, event_id=e.event_id, window_id=wid,
                                                  series_id=sid, n_obs=len(pair), status=status,
                                                  corr=pair.iloc[:, 0].corr(pair.iloc[:, 1])))

            # event path: every series indexed to 100 at t(-1) (rates: change in pp)
            lo, hi = PATH_RANGE
            base = t0 - 1
            for rel in range(lo, hi + 1):
                i = t0 + rel
                if i < 0 or i >= n:
                    continue
                rec = dict(basis=basis, event_id=e.event_id, rel_day=rel, date=dates[i].date())
                for sid in WINDOW_SERIES:
                    v, b = P[sid].iloc[i], P[sid].iloc[base]
                    if pd.isna(v) or pd.isna(b):
                        rec[sid] = np.nan
                    elif sid in LEVEL_CHANGE_SERIES:
                        rec[sid] = v - b
                    else:
                        rec[sid] = v / b * 100
                path_rows.append(rec)

    pd.DataFrame(anchors).to_csv(PROC / "event_anchor.csv", index=False)
    pd.DataFrame(rows).to_csv(PROC / "event_window.csv", index=False)
    pd.DataFrame(corr_rows).to_csv(PROC / "event_window_corr.csv", index=False)
    pd.DataFrame(ctx_rows).to_csv(PROC / "event_window_context.csv", index=False)
    path = pd.DataFrame(path_rows).melt(id_vars=["basis", "event_id", "rel_day", "date"],
                                        var_name="series_id", value_name="indexed_value")
    path.to_csv(PROC / "event_path.csv", index=False)
    w = pd.DataFrame(rows)
    print(w[(w.series_id == "USDINR") & (w.basis == "RBI")]
          .pivot(index="event_id", columns="window_id", values="change").round(2)
          [[x[0] for x in WINDOWS]].to_string())

In [6]:
compute_event_windows()

window_id  PRE_30  PRE_14  PRE_7  EVENT  POST_7  POST_14  POST_30  POST_90
event_id                                                                  
E01          0.88   -0.39  -0.08  -0.34   -1.58    -2.21    -2.19    -2.81
E02         -0.56   -0.00  -0.85   0.16    0.21     0.67     2.00     0.74
E03          0.20    0.33   0.01   0.41    0.05    -0.07    -0.47     1.48
E04          1.54   -0.27  -0.88   0.44    0.32    -1.42    -2.95    -2.94
E05         -0.64    0.88   0.21   0.49   -0.59    -0.14     0.06     6.02
E06          1.32    0.61   0.85  -0.35   -0.62    -1.87    -1.79    -3.32
E07          0.93   -0.31  -1.37   0.85    3.03     1.70     2.00     6.42
E08          0.83    0.22   0.02   0.02    0.02     0.03     0.12    -0.42
E09          0.61    0.55   0.04   0.06   -0.02     0.13    -0.11     0.69
E10         -2.61   -0.77  -1.75   0.40   -1.01     0.57     0.81     3.44
E11          0.57    0.75  -0.49   0.66    0.67    -0.03     1.16     2.96
E12          0.83    0.61

## What t(0) looks like for each event

In [7]:
anchors = pd.read_csv(PROC / "event_anchor.csv")
anchors[anchors.basis == "RBI"][["event_id", "anchor_date", "t0_date", "t_minus1_date", "trading_days_after_available"]]

,event_id,anchor_date,t0_date,t_minus1_date,trading_days_after_available
0,E01,2014-03-01,2014-03-03,2014-02-28,3029
1,E02,2016-09-29,2016-09-29,2016-09-28,2413
2,E03,2017-06-30,2017-06-30,2017-06-29,2232
3,E04,2019-02-15,2019-02-15,2019-02-14,1836
4,E05,2020-01-03,2020-01-03,2020-01-02,1625
5,E06,2020-06-16,2020-06-16,2020-06-15,1520
6,E07,2022-02-24,2022-02-24,2022-02-23,1108
7,E08,2023-10-09,2023-10-09,2023-10-06,717
8,E09,2024-04-15,2024-04-15,2024-04-12,595
9,E10,2025-04-23,2025-04-23,2025-04-22,349


## Headline table: USD/INR % change by window (RBI basis, + = rupee weaker)

In [8]:
ew = pd.read_csv(PROC / "event_window.csv")
order = [w[0] for w in WINDOWS]
usd = ew[(ew.basis == "RBI") & (ew.series_id == "USDINR")]
usd.pivot(index="event_id", columns="window_id", values="change")[order].round(2)

window_id,PRE_30,PRE_14,PRE_7,EVENT,POST_7,POST_14,POST_30,POST_90
event_id,,,,,,,,
E01,0.88,-0.39,-0.08,-0.34,-1.58,-2.21,-2.19,-2.81
E02,-0.56,-0.00,-0.85,0.16,0.21,0.67,2.00,0.74
E03,0.20,0.33,0.01,0.41,0.05,-0.07,-0.47,1.48
E04,1.54,-0.27,-0.88,0.44,0.32,-1.42,-2.95,-2.94
E05,-0.64,0.88,0.21,0.49,-0.59,-0.14,0.06,6.02
E06,1.32,0.61,0.85,-0.35,-0.62,-1.87,-1.79,-3.32
E07,0.93,-0.31,-1.37,0.85,3.03,1.70,2.00,6.42
E08,0.83,0.22,0.02,0.02,0.02,0.03,0.12,-0.42
E09,0.61,0.55,0.04,0.06,-0.02,0.13,-0.11,0.69


## Context: what else was happening in each +30-day window

In [9]:
ctx = pd.read_csv(PROC / "event_window_context.csv")
ctx[(ctx.basis == "RBI") & (ctx.window_id == "POST_30")][
    ["event_id", "start_date", "end_date", "rbi_repo_changes", "fomc_target_changes", "rbi_net_fx_usd_mn", "overlapping_events"]]

,event_id,start_date,end_date,rbi_repo_changes,fomc_target_changes,rbi_net_fx_usd_mn,overlapping_events
6,E01,2014-02-28,2014-04-22,NaN,NaN,13122.0,NaN
14,E02,2016-09-28,2016-11-16,2016-10-04 -25bp,NaN,1513.0,NaN
22,E03,2017-06-29,2017-08-11,2017-08-02 -25bp,NaN,9470.0,NaN
30,E04,2019-02-14,2019-04-04,2019-04-04 -25bp,NaN,15134.0,NaN
38,E05,2020-01-02,2020-02-14,NaN,NaN,19410.0,NaN
46,E06,2020-06-15,2020-07-28,NaN,NaN,25787.0,NaN
54,E07,2022-02-23,2022-04-12,NaN,2022-03-17 to 0.50%,-17365.0,NaN
62,E08,2023-10-06,2023-11-22,NaN,NaN,-2239.0,NaN
70,E09,2024-04-12,2024-05-31,NaN,NaN,575.0,NaN
78,E10,2025-04-22,2025-06-06,2025-06-06 -50bp,NaN,-3557.0,NaN


**Reading this table:** a large USD/INR move next to a repo change, an FOMC change or heavy RBI dollar sales is exactly
why these results are **temporal associations, not causal effects**.

**Output:** `event_window.csv`, `event_window_corr.csv`, `event_path.csv`, `event_window_context.csv`. Next: **04 · Database**.